# Radar de Vagas em Dados
### Dados por Todos · Ecossistema de Vagas

---

Este notebook sobe um **dashboard interativo** que monitora as vagas coletadas diariamente pelo scraper da comunidade **Dados por Todos**.

O pipeline completo:

```
GCP (Cloud Run + Scheduler)
        ↓  todo dia às 9h
   Scraper Python (requests + BeautifulSoup)
        ↓
   Google Sheets  ←── fonte de dados
        ↓
   Streamlit (este notebook) → ngrok (URL pública)
```

**Ordem obrigatória de execução:**
1. Célula 1 — Instalar dependências
2. Célula 2 — Autenticar no Google
3. Célula 3 — Gravar o app em disco
4. Célula 4 — Subir o dashboard

> ⚠️ Execute sempre nessa ordem a cada nova sessão do Colab.

---
## Célula 1 — Instalação de Dependências

Instala todos os pacotes necessários. Execute **uma vez por sessão**.

| Pacote | Para que serve |
|---|---|
| `streamlit` | Framework do dashboard |
| `pyngrok` | Cria URL pública temporária |
| `gspread` | Lê dados do Google Sheets |
| `google-auth` | Autenticação com a conta Google |
| `google-auth-oauthlib` | Suporte ao fluxo OAuth do Colab |
| `requests` | Requisições HTTP (dependência do gspread) |

In [1]:
# ─────────────────────────────────────────────────────────────
# CÉLULA 1 — DEPENDÊNCIAS
# Instala todos os pacotes. Execute uma vez por sessão.
# ─────────────────────────────────────────────────────────────

!pip install -q streamlit pyngrok gspread google-auth google-auth-oauthlib requests

print('✅ Dependências instaladas:')
print('   streamlit · pyngrok · gspread · google-auth · requests')

✅ Dependências instaladas:
   streamlit · pyngrok · gspread · google-auth · requests


---
## Célula 2 — Autenticação Google

Ao executar, aparece um popup pedindo para você escolher sua conta Google.

**O que acontece internamente:**
- O Colab cria um token OAuth **em memória** na sessão
- O `google.auth.default()` no app.py captura esse token automaticamente
- Nenhum arquivo de credenciais é salvo em disco (comportamento normal do Colab)

> ⚠️ Se pular esta etapa, o dashboard não conseguirá ler a planilha.

In [2]:
# ─────────────────────────────────────────────────────────────
# CÉLULA 2 — AUTENTICAÇÃO
# Abre o popup do Google. Execute ANTES do dashboard.
# ─────────────────────────────────────────────────────────────

from google.colab import auth

auth.authenticate_user()

print('✅ Autenticação concluída!')
print('   O Streamlit agora consegue acessar o Google Sheets.')

ModuleNotFoundError: No module named 'google.colab'

---
## Célula 3 — Código do Dashboard (app.py)

Esta célula **grava o arquivo `app.py`** no disco do Colab com a magic `%%writefile`.  
O Streamlit executará esse arquivo na próxima célula.

**Estrutura do app:**
```
app.py
├── Configuração da página
├── CSS — identidade visual Dados por Todos
├── load_data_from_sheets() — leitura do Sheets com logs
└── Render do dashboard
    ├── Header + banner de contexto
    ├── 4 KPI cards
    ├── Linha 1: tendência diária + modelo de trabalho
    ├── Linha 2: estados (UF) + top 10 cargos
    └── Tabela filtrável com todas as vagas
```

> 💡 Para trocar a planilha de origem, edite a variável `url` dentro de `load_data_from_sheets()`.

In [ ]:
%%writefile app.py

import streamlit as st
import pandas as pd
import plotly.express as px
import gspread
from google.auth import default

# ─────────────────────────────────────────────
# CONFIGURAÇÃO DA PÁGINA
# ─────────────────────────────────────────────
st.set_page_config(
    page_title="Dados por Todos · Radar de Vagas",
    page_icon="◆",
    layout="wide",
    initial_sidebar_state="collapsed"
)

st.markdown("""
<style>
    @import url('https://fonts.googleapis.com/css2?family=Montserrat:wght@300;400;600;700;800&display=swap');

    html, body, [class*="css"], .stApp {
        font-family: 'Montserrat', sans-serif !important;
        background-color: #1A1326 !important;
        color: #e8e0f5 !important;
    }
    .block-container {
        padding-top: 3.5rem !important;
        padding-bottom: 1.5rem !important;
        padding-left: 2.5rem !important;
        padding-right: 2.5rem !important;
        max-width: 1440px;
    }
    div[data-testid="stVerticalBlock"] > div { gap: 0.35rem !important; }
    ::-webkit-scrollbar { width: 4px; height: 4px; }
    ::-webkit-scrollbar-track { background: transparent; }
    ::-webkit-scrollbar-thumb { background: #917BFB44; border-radius: 4px; }
    .dpt-header {
        display: flex;
        align-items: flex-end;
        justify-content: space-between;
        padding-bottom: 0.75rem;
        border-bottom: 1px solid rgba(145, 123, 251, 0.2);
        margin-bottom: 0.25rem;
    }
    .dpt-tag {
        font-size: 0.72rem; font-weight: 700; letter-spacing: 0.22em;
        text-transform: uppercase; color: #917BFB; margin: 0 0 0.15rem 0;
    }
    .dpt-title {
        font-size: 1.6rem; font-weight: 800; letter-spacing: -0.025em;
        color: #e8e0f5; margin: 0; line-height: 1.1;
    }
    .dpt-title span { color: #917BFB; }
    .dpt-desc { font-size: 0.88rem; font-weight: 300; color: #AC91FA; margin: 0.2rem 0 0 0; opacity: 0.75; }
    .dpt-meta { text-align: right; font-size: 0.75rem; font-weight: 600; letter-spacing: 0.1em; text-transform: uppercase; color: #917BFB; opacity: 0.55; line-height: 1.7; }
    .context-banner {
        background: linear-gradient(135deg, rgba(145,123,251,0.07) 0%, rgba(206,94,253,0.04) 100%);
        border: 1px solid rgba(145, 123, 251, 0.15);
        border-left: 3px solid #917BFB;
        border-radius: 8px; padding: 0.55rem 1rem; margin: 0.4rem 0 0.5rem 0;
    }
    .context-banner p { margin: 0; font-size: 0.85rem; font-weight: 400; color: #CBFAFA; line-height: 1.6; opacity: 0.82; }
    .context-banner strong { color: #AC91FA; font-weight: 600; }
    .kpi-grid { display: grid; grid-template-columns: repeat(4, 1fr); gap: 0.55rem; margin: 0.4rem 0; }
    .kpi-card {
        background: rgba(145, 123, 251, 0.06); border: 1px solid rgba(145, 123, 251, 0.14);
        border-radius: 10px; padding: 0.8rem 1rem; position: relative; overflow: hidden;
    }
    .kpi-card::before {
        content: ''; position: absolute; top: 0; left: 0; right: 0; height: 2px;
        background: linear-gradient(90deg, #917BFB, #CE5EFD);
    }
    .kpi-label { font-size: 0.75rem; font-weight: 700; text-transform: uppercase; letter-spacing: 0.13em; color: #AC91FA; opacity: 0.65; margin-bottom: 0.25rem; }
    .kpi-value { font-size: 2.1rem; font-weight: 800; letter-spacing: -0.04em; color: #e8e0f5; line-height: 1; }
    .kpi-sub { font-size: 0.75rem; font-weight: 400; color: #AC91FA; opacity: 0.5; margin-top: 0.15rem; }
    .section-label {
        font-size: 0.72rem; font-weight: 700; text-transform: uppercase; letter-spacing: 0.18em;
        color: #917BFB; opacity: 0.6; display: flex; align-items: center; gap: 0.5rem; margin: 0.55rem 0 0.1rem 0;
    }
    .section-label::after { content: ''; flex: 1; height: 1px; background: rgba(145, 123, 251, 0.13); }
    .chart-title { font-size: 0.95rem; font-weight: 700; color: #e8e0f5; margin: 0 0 0.05rem 0; letter-spacing: -0.01em; }
    .chart-subtitle { font-size: 0.75rem; font-weight: 300; color: #AC91FA; opacity: 0.55; margin-bottom: 0.15rem; }
    div[data-testid="stPlotlyChart"] { margin-top: -0.4rem !important; }
    div[data-testid="stDataFrame"] { margin-top: -0.2rem; border-radius: 8px; overflow: hidden; border: 1px solid rgba(145, 123, 251, 0.15) !important; }
    div[data-testid="stDataFrame"] th { font-family: 'Montserrat', sans-serif !important; font-size: 0.78rem !important; font-weight: 700 !important; text-transform: uppercase; letter-spacing: 0.08em; color: #917BFB !important; }
    div[data-testid="stDataFrame"] td { font-family: 'Montserrat', sans-serif !important; font-size: 0.88rem !important; }
    div[data-testid="stButton"] button { font-family: 'Montserrat', sans-serif !important; font-size: 0.82rem; font-weight: 700; letter-spacing: 0.08em; background: rgba(145, 123, 251, 0.1); border: 1px solid rgba(145, 123, 251, 0.28); color: #AC91FA; border-radius: 6px; }
    div[data-testid="stButton"] button:hover { background: rgba(145, 123, 251, 0.2); border-color: #917BFB; color: #e8e0f5; }
    div[data-testid="stSelectbox"] > div, div[data-testid="stTextInput"] > div > div { font-family: 'Montserrat', sans-serif !important; font-size: 0.88rem !important; background: rgba(145, 123, 251, 0.05) !important; border-color: rgba(145, 123, 251, 0.18) !important; border-radius: 6px !important; }
    div[data-testid="stCaptionContainer"] p { font-family: 'Montserrat', sans-serif !important; font-size: 0.78rem !important; color: #AC91FA !important; opacity: 0.55; }
    div[data-testid="metric-container"] { display: none; }
    #MainMenu, footer, header { visibility: hidden; }
    div[data-testid="stDecoration"] { display: none; }
</style>
""", unsafe_allow_html=True)


# ─────────────────────────────────────────────
# CARREGAMENTO DOS DADOS
# A autenticação usa google.auth.default() que no Colab
# captura automaticamente o token da sessão após
# auth.authenticate_user() ter sido chamado na mesma sessão.
# ─────────────────────────────────────────────
@st.cache_data(ttl=600)
def load_data_from_sheets():
    print("\n" + "="*60)
    print("[INICIO] load_data_from_sheets()")
    print("="*60)

    try:
        # No Colab, google.auth.default() pega o token da sessão
        # ativo após auth.authenticate_user() — sem arquivo em disco.
        print("\n[AUTH] Obtendo credenciais via google.auth.default()...")
        SCOPES = [
            "https://spreadsheets.google.com/feeds",
            "https://www.googleapis.com/auth/drive",
        ]
        creds, project = default(scopes=SCOPES)
        print(f"[AUTH] Credenciais obtidas! Tipo: {type(creds).__name__}")
        print(f"[AUTH] Token valido: {creds.valid}")
        print(f"[AUTH] Projeto: {project}")

        # Conexão gspread
        print("\n[GSPREAD] Conectando...")
        client = gspread.authorize(creds)
        print("[GSPREAD] Conectado!")

        # Planilha
        url = 'https://docs.google.com/spreadsheets/d/1xvuHa4a-vOTCQemrV9giBtAc2Z81rs96Dm87KsoLq1E/edit?usp=sharing'
        print(f"\n[SHEETS] Abrindo planilha...")
        sh = client.open_by_url(url)
        print(f"[SHEETS] Planilha: '{sh.title}'")
        print(f"[SHEETS] Abas: {[ws.title for ws in sh.worksheets()]}")

        # Leitura
        worksheet = sh.get_worksheet(0)
        print(f"\n[LEITURA] Aba: '{worksheet.title}'")
        data = worksheet.get_all_values()
        print(f"[LEITURA] Linhas lidas (com header): {len(data)}")
        print(f"[LEITURA] Colunas: {data[0] if data else '(vazio)'}")

        if len(data) < 2:
            print("[LEITURA] ERRO: planilha vazia!")
            st.warning("A planilha está vazia.")
            return pd.DataFrame()

        # DataFrame
        df = pd.DataFrame(data[1:], columns=data[0])
        print(f"\n[DF] Shape: {df.shape}")
        print(f"[DF] Colunas: {list(df.columns)}")

        # Datas
        df['date_published'] = pd.to_datetime(df['date_published'], errors='coerce')
        nulos = df['date_published'].isna().sum()
        print(f"\n[DATAS] Invalidas: {nulos}")
        df = df.dropna(subset=['date_published'])
        print(f"[DATAS] Shape final: {df.shape}")
        print(f"[DATAS] Range: {df['date_published'].min().date()} -> {df['date_published'].max().date()}")

        # Limpeza
        df['site_source'] = df['site_source'].str.lower().str.strip()
        print(f"\n[LIMPEZA] Fontes: {df['site_source'].value_counts().to_dict()}")
        df['tipo_vaga'] = df['tipo_vaga'].replace('', 'Não Informado').fillna('Não Informado')
        print(f"[LIMPEZA] Tipos: {df['tipo_vaga'].value_counts().to_dict()}")
        df['location'] = df['location'].replace('', 'Não Informada').fillna('Não Informada')

        # Filtro Engenha
        print(f"\n[FILTRO] Antes: {len(df)} linhas")
        df_engenha = df[df['site_source'] == 'engenha'].copy()
        print(f"[FILTRO] Depois (engenha): {len(df_engenha)} linhas")

        if df_engenha.empty:
            print(f"[FILTRO] ERRO! Fontes disponiveis: {df['site_source'].unique().tolist()}")
            st.warning("Nenhuma vaga encontrada para 'engenha'.")
            return pd.DataFrame()

        # UF
        def extrair_uf(loc):
            if loc == 'Não Informada' or pd.isna(loc):
                return 'N/I'
            partes = str(loc).split(',')
            if len(partes) > 1:
                uf = partes[-1].strip().upper()
                if len(uf) == 2:
                    return uf
            return 'N/I'

        df_engenha['estado_uf'] = df_engenha['location'].apply(extrair_uf)
        uf_dist = df_engenha['estado_uf'].value_counts().to_dict()
        ni = uf_dist.pop('N/I', 0)
        print(f"\n[UF] Com UF: {sum(uf_dist.values())} | Sem UF: {ni}")
        print(f"[UF] Top estados: {dict(list(uf_dist.items())[:8])}")

        if 'extracted_query' in df_engenha.columns:
            print(f"\n[QUERY] Top 5: {df_engenha['extracted_query'].value_counts().head(5).to_dict()}")
        else:
            print(f"\n[QUERY] Coluna ausente. Disponíveis: {list(df_engenha.columns)}")

        print("\n" + "="*60)
        print(f"[FIM] {len(df_engenha)} vagas carregadas!")
        print("="*60 + "\n")

        return df_engenha

    except Exception as e:
        import traceback
        print("\n" + "!"*60)
        print(f"[ERRO] {type(e).__name__}: {e}")
        print(traceback.format_exc())
        print("!"*60 + "\n")
        st.error(f"Erro ao conectar com a planilha: {e}")
        return pd.DataFrame()


df = load_data_from_sheets()

DPT_MAIN   = '#917BFB'
DPT_ACCENT = '#CE5EFD'
DPT_TEAL   = '#CBFAFA'
DPT_SCALE  = [[0, '#2D1F4A'], [0.5, '#917BFB'], [1, '#CE5EFD']]
DPT_SCALE2 = [[0, '#2D1F4A'], [0.5, '#917BFB'], [1, '#CBFAFA']]

CHART_LAYOUT = dict(
    template="plotly_dark",
    margin=dict(l=0, r=0, t=4, b=0),
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    font=dict(family="Montserrat, sans-serif", size=11, color="#AC91FA"),
    height=215,
)

if not df.empty:

    col_title, col_btn = st.columns([9, 1])
    with col_title:
        ultima_data_str = df['date_published'].max().strftime('%d/%m/%Y')
        st.markdown(f"""
        <div class="dpt-header">
            <div>
                <p class="dpt-tag">Dados por Todos &nbsp;·&nbsp; Ecossistema de Vagas</p>
                <p class="dpt-title">Radar de <span>Vagas em Dados</span></p>
                <p class="dpt-desc">Monitoramento diário de oportunidades coletadas via scraper — base para o bot de vagas da comunidade</p>
            </div>
            <div class="dpt-meta">Fonte &nbsp;·&nbsp; Engenha<br>Atualizado em {ultima_data_str}</div>
        </div>
        """, unsafe_allow_html=True)
    with col_btn:
        st.markdown("<div style='height:2.8rem'></div>", unsafe_allow_html=True)
        if st.button("Atualizar", use_container_width=True):
            st.cache_data.clear()
            st.rerun()

    st.markdown("""
    <div class="context-banner">
        <p>
            Este painel acompanha as vagas coletadas diariamente pelo scraper da <strong>Dados por Todos</strong> — uma comunidade criada para conectar profissionais e carreiras na área de dados.
            Todo dia às <strong>9h</strong>, as oportunidades mais recentes são enviadas automaticamente por e-mail para curadoria e distribuição no grupo de vagas.
            Em breve, este fluxo alimentará um <strong>bot integrado ao ecossistema da comunidade</strong>, tornando a entrega de vagas ainda mais ágil e precisa.
        </p>
    </div>
    """, unsafe_allow_html=True)

    total         = len(df)
    remotas       = len(df[df['tipo_vaga'].str.contains('Remoto', case=False, na=False)])
    pct_remoto    = (remotas / total * 100) if total > 0 else 0
    df_uf_validos = df[df['estado_uf'] != 'N/I']
    top_uf        = df_uf_validos['estado_uf'].mode()[0] if not df_uf_validos.empty else "N/A"
    ultima_data   = df['date_published'].max().strftime('%d/%m/%Y')
    vagas_hoje    = len(df[df['date_published'].dt.date == df['date_published'].dt.date.max()])

    st.markdown(f"""
    <div class="kpi-grid">
        <div class="kpi-card">
            <div class="kpi-label">Total Coletado</div>
            <div class="kpi-value">{total:,}</div>
            <div class="kpi-sub">vagas no banco de dados</div>
        </div>
        <div class="kpi-card">
            <div class="kpi-label">Novas Hoje</div>
            <div class="kpi-value">{vagas_hoje:,}</div>
            <div class="kpi-sub">publicadas em {ultima_data}</div>
        </div>
        <div class="kpi-card">
            <div class="kpi-label">Trabalho Remoto</div>
            <div class="kpi-value">{pct_remoto:.1f}%</div>
            <div class="kpi-sub">{remotas} vagas remotas</div>
        </div>
        <div class="kpi-card">
            <div class="kpi-label">Estado com Mais Vagas</div>
            <div class="kpi-value">{top_uf}</div>
            <div class="kpi-sub">maior concentração de oportunidades</div>
        </div>
    </div>
    """, unsafe_allow_html=True)

    st.markdown('<div class="section-label">Tendência e Distribuição</div>', unsafe_allow_html=True)
    c1, c2 = st.columns([3, 2])

    with c1:
        df_time = df.groupby(df['date_published'].dt.date).size().reset_index(name='Vagas')
        fig_time = px.area(df_time, x='date_published', y='Vagas', labels={'date_published': '', 'Vagas': ''})
        fig_time.update_traces(line_shape='spline', line_color=DPT_MAIN, fillcolor='rgba(145, 123, 251, 0.11)')
        fig_time.update_layout(**CHART_LAYOUT, hovermode="x unified",
            xaxis=dict(showgrid=False, showline=False, tickfont=dict(size=10)),
            yaxis=dict(showgrid=False, showline=False, tickfont=dict(size=10)))
        st.markdown('<p class="chart-title">Volume de Publicações por Dia</p>', unsafe_allow_html=True)
        st.markdown('<p class="chart-subtitle">Histórico completo desde o início da coleta</p>', unsafe_allow_html=True)
        st.plotly_chart(fig_time, use_container_width=True, config={'displayModeBar': False})

    with c2:
        df_tipo = df['tipo_vaga'].value_counts().reset_index()
        df_tipo.columns = ['Modelo', 'Qtd']
        fig_tipo = px.bar(df_tipo, x='Qtd', y='Modelo', orientation='h', color='Qtd', color_continuous_scale=DPT_SCALE)
        fig_tipo.update_layout(**CHART_LAYOUT,
            yaxis=dict(categoryorder='total ascending', showgrid=False, tickfont=dict(size=10)),
            xaxis=dict(showgrid=False, tickfont=dict(size=10)), coloraxis_showscale=False)
        st.markdown('<p class="chart-title">Modelo de Trabalho</p>', unsafe_allow_html=True)
        st.markdown('<p class="chart-subtitle">Presencial, híbrido e remoto</p>', unsafe_allow_html=True)
        st.plotly_chart(fig_tipo, use_container_width=True, config={'displayModeBar': False})

    st.markdown('<div class="section-label">Geografia e Termos Mais Buscados</div>', unsafe_allow_html=True)
    c3, c4 = st.columns([3, 2])

    with c3:
        df_uf_contagem = df[df['estado_uf'] != 'N/I']['estado_uf'].value_counts().reset_index()
        df_uf_contagem.columns = ['Estado', 'Vagas']
        fig_uf = px.bar(df_uf_contagem, x='Estado', y='Vagas', text_auto=True, color='Vagas', color_continuous_scale=DPT_SCALE2)
        fig_uf.update_layout(**CHART_LAYOUT, coloraxis_showscale=False,
            xaxis=dict(showgrid=False, tickfont=dict(size=9)),
            yaxis=dict(showgrid=False, tickfont=dict(size=10)))
        fig_uf.update_traces(textfont_size=8, textposition='outside', textfont_color='#AC91FA')
        st.markdown('<p class="chart-title">Vagas por Estado (UF)</p>', unsafe_allow_html=True)
        st.markdown('<p class="chart-subtitle">Distribuição geográfica das oportunidades</p>', unsafe_allow_html=True)
        st.plotly_chart(fig_uf, use_container_width=True, config={'displayModeBar': False})

    with c4:
        if 'extracted_query' in df.columns:
            query_counts = df['extracted_query'].value_counts().nlargest(10).reset_index()
            query_counts.columns = ['Termo', 'Vol']
            fig_query = px.bar(query_counts, x='Vol', y='Termo', orientation='h', color_discrete_sequence=[DPT_TEAL])
            fig_query.update_layout(**CHART_LAYOUT,
                yaxis=dict(categoryorder='total ascending', showgrid=False, tickfont=dict(size=9)),
                xaxis=dict(showgrid=False, tickfont=dict(size=10)))
            st.markdown('<p class="chart-title">Top 10 Cargos Monitorados</p>', unsafe_allow_html=True)
            st.markdown('<p class="chart-subtitle">Perfis mais rastreados pelo scraper</p>', unsafe_allow_html=True)
            st.plotly_chart(fig_query, use_container_width=True, config={'displayModeBar': False})

    st.markdown('<div class="section-label">Banco de Oportunidades</div>', unsafe_allow_html=True)

    f1, f2, f3, _ = st.columns([2, 2, 2, 4])
    with f1:
        ufs = ['Todos'] + sorted(df['estado_uf'].unique().tolist())
        filtro_uf = st.selectbox("UF", ufs, label_visibility="collapsed")
    with f2:
        tipos = ['Todos'] + sorted(df['tipo_vaga'].unique().tolist())
        filtro_tipo = st.selectbox("Modelo", tipos, label_visibility="collapsed")
    with f3:
        filtro_busca = st.text_input("cargo", placeholder="Buscar por cargo...", label_visibility="collapsed")

    df_view = df[['title', 'company', 'location', 'estado_uf', 'tipo_vaga', 'date_published', 'url']].copy()
    df_view = df_view.sort_values('date_published', ascending=False)

    if filtro_uf   != 'Todos': df_view = df_view[df_view['estado_uf'] == filtro_uf]
    if filtro_tipo != 'Todos': df_view = df_view[df_view['tipo_vaga'] == filtro_tipo]
    if filtro_busca:            df_view = df_view[df_view['title'].str.contains(filtro_busca, case=False, na=False)]

    st.caption(f"{len(df_view)} oportunidades encontradas")
    st.dataframe(
        df_view,
        column_config={
            "title":          st.column_config.TextColumn("Cargo", width="large"),
            "company":        st.column_config.TextColumn("Empresa", width="medium"),
            "location":       st.column_config.TextColumn("Localização", width="medium"),
            "estado_uf":      st.column_config.TextColumn("UF", width="small"),
            "tipo_vaga":      st.column_config.TextColumn("Modelo", width="medium"),
            "date_published": st.column_config.DateColumn("Publicado", format="DD/MM/YY", width="small"),
            "url":            st.column_config.LinkColumn("Acessar Vaga", width="small"),
        },
        use_container_width=True,
        hide_index=True,
        height=320,
    )

else:
    st.warning("Nenhum dado encontrado. Verifique a conexão com a planilha.")


---
## Célula 4 — Subir o Dashboard

Esta célula configura o ngrok e inicia o Streamlit.

**O que é o ngrok?**  
O Streamlit roda localmente no servidor do Colab. O ngrok cria um túnel que expõe essa porta com uma URL pública — válida enquanto a sessão estiver ativa.

> ⚠️ A URL expira quando a sessão do Colab encerrar.  
> 🔑 Crie seu token gratuito em [ngrok.com](https://ngrok.com) e substitua abaixo.

In [ ]:
# ─────────────────────────────────────────────────────────────
# CÉLULA 4 — SUBIR O DASHBOARD
# Execute após as células 1, 2 e 3.
# ─────────────────────────────────────────────────────────────

from pyngrok import ngrok

# ⚠️ Substitua pelo seu token: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN = '3CHldPp0PSQEcsBjPwBAd3p40O9_5JtPLDy3iQkFFCzSShF9n'

ngrok.set_auth_token(NGROK_TOKEN)

# Cria o túnel na porta padrão do Streamlit
public_url = ngrok.connect(8501, proto='http')
print(f'✅ Dashboard disponível em: {public_url}')
print('   Abra o link acima no navegador.')
print('   O link expira quando a sessão do Colab encerrar.')

# Inicia o Streamlit (bloqueia a célula enquanto o app roda)
!streamlit run app.py

---
## Troubleshooting

| Erro | Causa | Solução |
|---|---|---|
| `Nenhum dado encontrado` | Autenticação falhou | Re-execute Célula 2 e depois Célula 4 |
| `ModuleNotFoundError` | Pacote não instalado | Re-execute Célula 1 |
| `ngrok error` | Token inválido | Verifique seu token em ngrok.com |
| Dashboard em branco | Streamlit ainda iniciando | Aguarde ~10s e recarregue |
| Dados desatualizados | Cache ativo (10 min) | Clique em Atualizar no dashboard |

---
*Dados por Todos · comunidade de profissionais de dados*